In [1]:
#%pip install transformers pillow
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
import torch.nn as nn
from transformers import AutoProcessor, AutoModel
from PIL import Image


In [2]:
import sys
import os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    load_dir = '/content/drive/MyDrive/xai-project5/results/feature_extraction'
    scripts_path = '/content/drive/MyDrive/xai-project5/src/scripts'
else:
    load_dir = os.path.abspath(os.path.join('..', 'results', 'feature_extraction'))
    scripts_path = os.path.abspath('../scripts')

if scripts_path not in sys.path:
    sys.path.append(scripts_path)

load_path = os.path.join(load_dir, 'nih_chest_embeddings.pt')
print(f"Caricamento configurato da: {load_path}")

Gli **Sparse Autoencoders (SAE)** implementano una forma di *sparse dictionary learning*, con l'obiettivo di apprendere una decomposizione sparsa di un segnale in un dizionario sovraccompleto di atomi.

---

Un SAE è costituito da:
- **Encoder** $W_{enc} \in \mathbb{R}^{d \times \omega}$: trasforma l'embedding di input in uno spazio latente
- **Decoder** $W_{dec} \in \mathbb{R}^{\omega \times d}$: ricostruisce l'embedding originale dallo spazio latente
- **Funzione di attivazione non-lineare** $\sigma : \mathbb{R}^{\omega} \to \mathbb{R}^{\omega}$
- **Bias condiviso** $b \in \mathbb{R}^d$: sottratto dall'input dell'encoder e aggiunto all'output del decoder

La larghezza dello strato latente $\omega$ è scelta come fattore della dimensione originale: $\omega := d \times \varepsilon$, dove $\varepsilon$ è il **fattore di espansione**.

---

Dato un embedding $v \in \mathbb{R}^d$, il SAE decompose il vettore in:
- **Vettore di attivazioni**: $\phi(v) := \sigma(W_{enc}^{\top}(v - b))$
- **Vettore ricostruito**: $\hat{v} := W_{dec}^{\top}\phi(v) + b$

---

La loss function combina un **obiettivo di ricostruzione** con una **regolarizzazione di sparsità**:

$$\mathcal{L}(v) = R(v) + \lambda S(v)$$

dove:
- **Ricostruzione (L2)**: $R(v) := \|v - \hat{v}\|_2^2$ garantisce la fedeltà dell'informazione
- **Sparsità (L1)**: $S(v) := \|\phi(v)\|_1$ penalizza l'attivazione di troppi neuroni latenti
- **Hyperparameter** $\lambda$: regola il trade-off tra ricostruzione e sparsità


---

L'implementazione completa del SAE è disponibile nel file `scripts/sae.py`, dove sono definiti:
- La classe `SparseAutoencoder` con i metodi `forward()` e `normalize_decoder_weights()`
- La funzione di loss `sae_loss_function()` che calcola il trade-off tra ricostruzione e sparsità

## Estrazione degli Embedding con CLIP

In questa sezione, utilizziamo il modello **PubMed CLIP** ([flaviagiammarino/pubmed-clip-vit-base-patch32](https://huggingface.co/flaviagiammarino/pubmed-clip-vit-base-patch32)) per estrarre gli embedding visivi dalle immagini mediche.

PubMed CLIP è una variante specializzata del modello CLIP (Contrastive Language-Image Pre-training) addestrata specificamente su dati biomedici. Questo modello fornisce:

- **Vision Encoder (ViT-Base)**: Trasforma le immagini mediche in embedding vettoriali di dimensione 512
- **Text Encoder**: Trasforma descrizioni testuali in embedding dello stesso spazio latente
- **Allineamento multimodale**: Gli embedding visivi e testuali sono proiettati nello stesso spazio, permettendo il calcolo della similarità coseno

Gli embedding estratti sono **normalizzati L2**, garantendo che ogni vettore abbia norma unitaria. Questo è fondamentale per:
1. Stabilizzare il training dello Sparse Autoencoder
2. Semplificare il calcolo della similarità coseno (che diventa una semplice moltiplicazione matriciale)
3. Interpretare semanticamente i neuroni latenti del SAE

In [3]:
model_id = "flaviagiammarino/pubmed-clip-vit-base-patch32"

print(f"Scaricamento del modello {model_id} in corso...")
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModel.from_pretrained(model_id)

Scaricamento del modello flaviagiammarino/pubmed-clip-vit-base-patch32 in corso...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Il codice sotto è stato usato solo per provare come il comportamento con un immagine

In [ ]:
#model.eval()
#image = Image.open("../images/raggi_x.jpg")

#if image.mode != "RGB":
#    image = image.convert("RGB")

#inputs = processor(images=image, return_tensors="pt")

#with torch.no_grad():
    # Il modello restituisce l'oggetto contenitore
#    vision_outputs = model.get_image_features(**inputs)
#vision_tensor = vision_outputs.pooler_output
    # Estraiamo il tensore

#vision_embeddings = F.normalize(vision_tensor, p=2, dim=1)

#print("Shape dell'embedding estratto:", vision_embeddings.shape)

In [4]:
%load_ext autoreload
%autoreload 2
import torch
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sae import SparseAutoencoder, sae_loss_function

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Utilizzando il device: {device}")

print("Carichiamo il dataset presente in nih_chest_embeddings.pt...")
vision_embeddings=torch.load(load_path).to(device)
dataset= TensorDataset(vision_embeddings)

batch_size=256
dataloader=DataLoader(dataset,batch_size=batch_size,shuffle=True)

sae = SparseAutoencoder(input_dim=512, hidden_dim=2048).to(device)
learning_rate = 1e-3
optimizer = optim.Adam(sae.parameters(), lr=learning_rate)

l1_lambda = 1e-4
num_epochs = 20
print("ADDESTRAMENTO SAE...")
for epoch in range(num_epochs):
    sae.train()
    epoch_total_loss=0.0
    epoch_mse_loss=0.0
    epoch_l1_loss=0.0

    for batch in dataloader:
        x=batch[0].to(device)
        #forward 
        x_hat,z=sae(x)

        #calcolo loss
        total_loss, mse_loss, l1_loss = sae_loss_function(x, x_hat, z, l1_lambda)
        #backward e ottimizzazione
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

        sae.normalize_decoder_weights() #si forza ad 1 per evitare il collasso

        #aggiorniamo le metriche
        epoch_total_loss += total_loss.item()
        epoch_mse_loss += mse_loss.item()
        epoch_l1_loss += l1_loss.item()
    
    avg_total_loss = epoch_total_loss / len(dataloader)
    avg_mse = epoch_mse_loss / len(dataloader)
    avg_l1 = epoch_l1_loss / len(dataloader)

    if (epoch + 1) % 5 == 0:
        print(f"Epoca [{epoch+1:02d}/{num_epochs}] | "
              f"Loss Tot: {avg_total_loss:.4f} | "
              f"MSE (Ricostruzione): {avg_mse:.4f} | "
              f"L1 (Sparsità): {avg_l1:.4f}")

Utilizzando il device: cpu
Carichiamo il dataset presente in nih_chest_embeddings.pt...
ADDESTRAMENTO SAE...
Epoca [05/20] | Loss Tot: 0.0001 | MSE (Ricostruzione): 0.0001 | L1 (Sparsità): 0.0047
Epoca [10/20] | Loss Tot: 0.0001 | MSE (Ricostruzione): 0.0001 | L1 (Sparsità): 0.0047
Epoca [15/20] | Loss Tot: 0.0000 | MSE (Ricostruzione): 0.0000 | L1 (Sparsità): 0.0048
Epoca [20/20] | Loss Tot: 0.0000 | MSE (Ricostruzione): 0.0000 | L1 (Sparsità): 0.0048




L'addestramento non supervisionato del nostro *Sparse Autoencoder* (SAE) ha prodotto un dizionario sovracompleto di feature: la matrice dei pesi del decoder, $W_{dec} \in \mathbb{R}^{512 \times 2048}$. Ogni colonna di questa matrice rappresenta un singolo "concetto visivo" che il modello ha isolato autonomamente guardando i pixel. Tuttavia, questi concetti sono matematicamente "muti": il modello ne riconosce l'esistenza, ma non possiede un'etichetta semantica umana per descriverli.

L'assegnazione delle etichette (Grounding) avviene confrontando i concetti del SAE con gli embedding testuali. Poiché entrambi i tensori sono stati preventivamente assoggettati a **normalizzazione L2** (rendendo i vettori di norma unitaria), il calcolo della similarità coseno si riduce a una singola moltiplicazione matriciale:

$$S = T \cdot W_{dec}$$

Dove $T$ è la matrice degli embedding testuali e $W_{dec}$ è il dizionario del SAE. Estraendo i valori massimi (*Top-K*) da questo prodotto, identifichiamo in modo inequivocabile quali specifici neuroni dell'Autoencoder si sono specializzati nel rilevare determinati concetti medici, rendendo la rappresentazione finale totalmente interpretabile.

In [5]:
#i nostri concetti 
medical_concepts = [
    "healthy lungs", 
    "bone fracture", 
    "pneumonia", 
    "pleural effusion",
    "heart",
    "ribs",
    "medical imaging artifact"
]

text_inputs=processor(text=medical_concepts,padding=True,return_tensors='pt')

with torch.no_grad():
    text_outputs=model.get_text_features(**text_inputs)
    text_tensor = text_outputs.pooler_output
#normalizziamo i concetti (N_concetti, 512)
text_embeddings=F.normalize(text_tensor, p=2,dim=1).to(device)

with torch.no_grad():
    sae_dictionary=F.normalize(sae.decoder.weight.data,p=2,dim=0)  

similarities=torch.matmul(text_embeddings,sae_dictionary) #calcoliamo la similarità 

top_k = 3
for idx, concept in enumerate(medical_concepts):
    concept_sims = similarities[idx]
    # Otteniamo i primi k valori e i loro indici (i "neuroni" del SAE)
    top_values, top_indices = torch.topk(concept_sims, top_k)
    
    print(f"\nConcetto testuale: '{concept}'")
    for i in range(top_k):
        print(f"  -> Neurone SAE {top_indices[i].item():4d} (Similarità: {top_values[i].item():.4f})")


Concetto testuale: 'healthy lungs'
  -> Neurone SAE 1342 (Similarità: 0.1725)
  -> Neurone SAE  744 (Similarità: 0.1470)
  -> Neurone SAE  183 (Similarità: 0.1436)

Concetto testuale: 'bone fracture'
  -> Neurone SAE 1024 (Similarità: 0.1549)
  -> Neurone SAE 1740 (Similarità: 0.1451)
  -> Neurone SAE 1342 (Similarità: 0.1378)

Concetto testuale: 'pneumonia'
  -> Neurone SAE  744 (Similarità: 0.1590)
  -> Neurone SAE 1120 (Similarità: 0.1275)
  -> Neurone SAE  253 (Similarità: 0.1274)

Concetto testuale: 'pleural effusion'
  -> Neurone SAE  730 (Similarità: 0.1450)
  -> Neurone SAE 1617 (Similarità: 0.1434)
  -> Neurone SAE 1024 (Similarità: 0.1403)

Concetto testuale: 'heart'
  -> Neurone SAE  544 (Similarità: 0.1564)
  -> Neurone SAE 1342 (Similarità: 0.1370)
  -> Neurone SAE 1740 (Similarità: 0.1369)

Concetto testuale: 'ribs'
  -> Neurone SAE 1740 (Similarità: 0.1574)
  -> Neurone SAE  544 (Similarità: 0.1470)
  -> Neurone SAE 1521 (Similarità: 0.1376)

Concetto testuale: 'medical